In [0]:
df_silver = spark.table("workspace.default.silver_diabetic")

print(f"Rows: {df_silver.count()}")
print(f"Columns: {len(df_silver.columns)}")

Rows: 101763
Columns: 49


In [0]:
from pyspark.sql.functions import col, when

df_gold = df_silver

# Total prior hospital visits
df_gold = df_gold.withColumn(
    "total_prior_visits",
    col("number_outpatient") + col("number_emergency") + col("number_inpatient")
)

# Ratio of procedures per day in hospital
df_gold = df_gold.withColumn(
    "procedures_per_day",
    col("num_procedures") / col("time_in_hospital")
)

# Ratio of medications per day in hospital
df_gold = df_gold.withColumn(
    "medications_per_day",
    col("num_medications") / col("time_in_hospital")
)

# Flag if patient has emergency history
df_gold = df_gold.withColumn(
    "has_emergency_history",
    when(col("number_emergency") > 0, 1).otherwise(0)
)

# Flag if patient has prior inpatient visits
df_gold = df_gold.withColumn(
    "has_prior_inpatient",
    when(col("number_inpatient") > 0, 1).otherwise(0)
)

print("Features created successfully!")
df_gold.select(
    "total_prior_visits",
    "procedures_per_day",
    "medications_per_day",
    "has_emergency_history",
    "has_prior_inpatient"
).describe().display()

Features created successfully!


summary,total_prior_visits,procedures_per_day,medications_per_day,has_emergency_history,has_prior_inpatient
count,101763,101763,101763,101763,101763
mean,1.202794728928982,0.4526605008806285,5.0743139662098296,0.11185794443953107,0.3354460855124161
stddev,2.291805499870224,0.8722743451087663,3.825717777759551,0.3151931486582683,0.472148493429935
min,0,0.0,0.08333333333333333,0,0
max,80,6.0,42.0,1,1


In [0]:
print("readmitted_flag" in df_gold.columns)

True


In [0]:
feature_columns = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_diagnoses",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "total_prior_visits",
    "procedures_per_day",
    "medications_per_day",
    "has_emergency_history",
    "has_prior_inpatient",
    "readmitted_flag"
]

df_gold_final = df_gold.select(feature_columns)

print(f"Final dataset: {df_gold_final.count()} rows, {len(df_gold_final.columns)} columns")
df_gold_final.display()

Final dataset: 101763 rows, 14 columns


time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_diagnoses,number_outpatient,number_emergency,number_inpatient,total_prior_visits,procedures_per_day,medications_per_day,has_emergency_history,has_prior_inpatient,readmitted_flag
1,41,0,1,1,0,0,0,0,0.0,1.0,0,0,0
3,59,0,18,9,0,0,0,0,0.0,6.0,0,0,0
2,11,5,13,6,2,0,1,3,2.5,6.5,0,1,0
2,44,1,16,7,0,0,0,0,0.5,8.0,0,0,0
1,51,0,8,5,0,0,0,0,0.0,8.0,0,0,0
3,31,6,16,9,0,0,0,0,2.0,5.333333333333333,0,0,0
4,70,1,21,7,0,0,0,0,0.25,5.25,0,0,0
5,73,0,12,8,0,0,0,0,0.0,2.4,0,0,0
13,68,2,28,8,0,0,0,0,0.15384615384615385,2.1538461538461537,0,0,0
12,33,3,18,8,0,0,0,0,0.25,1.5,0,0,0


In [0]:
(
    df_gold_final.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.gold_diabetic")
)

print("Gold saved successfully!")

Gold saved successfully!
